# Домашнее задание: учим сеть угадывать рисунки

На семинаре мы учили сеть распознавать цифры. В домашке возьмём данные поживее: рисунки из игры [Quick, Draw!](https://quickdraw.withgoogle.com/) от Google. Там игроку дают слово, у него 20 секунд, чтобы это нарисовать, а сеть пытается угадать, что он рисует. Google выложил в открытый доступ больше 50 миллионов таких рисунков.

Мы возьмём 10 классов (яблоко, банан, кот, рыба, гитара, дом, луна, самолёт, велосипед, зонтик), по 2500 картинок на класс. Каждая картинка 28 на 28 пикселей, так что по размеру всё как с цифрами. Но рисуют люди очень по-разному и часто как попало, поэтому задача заметно интереснее.

Основа такая же, как на семинаре: свой Dataset, сеть из линейных слоёв, цикл обучения. Поверх этого добавятся три новые вещи:
1. посчитать градиент слоя руками и сверить с тем, что выдаёт PyTorch;
2. сравнить SGD, SGD с momentum и Adam;
3. переобучить сеть специально, а потом лечить её через Dropout и BatchNorm.

Код пишете там, где стоит `# ваш код`. Остальные ячейки просто запускаете. На вопросы отвечайте в ячейках «Ответы» своими словами, хватит 2-3 предложений на вопрос.

### Оценивание

| Часть | Что смотрим | Баллы |
|---|---|---|
| 1. Pipeline | Dataset и DataLoader, модель, функции `train_model` и `accuracy`, ответы на вопросы | 2.5 |
| 2. Градиент руками | формулы, `manual_grads` без `.backward()`, сверка с autograd, проверка на слое обученной сети | 2.5 |
| 3. Оптимизаторы | эксперимент с тремя оптимизаторами, графики сходимости, выводы | 2.5 |
| 4. Dropout и BatchNorm | модель `DrawNetReg`, четыре эксперимента с графиками, эксперимент с `model.train()`, анализ переобучения | 2.5 |

**Пересдачи нет.** Поэтому перед отправкой сделайте `Kernel > Restart & Run All` и проверьте, что всё выполняется сверху вниз без ошибок. Если ячейка падает, задание из неё не засчитывается. Сдаёте `.ipynb` с выводами и графиками.


In [ ]:
import os
import json
import urllib.request

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.utils.data as data
import torchvision.transforms.v2 as tfs

## Данные

Ячейка ниже скачивает рисунки и раскладывает их по папкам в том же формате, что был на семинаре: `quickdraw/train/<класс>/*.png`, `quickdraw/test/<класс>/*.png` и `format.json` с номерами классов. Скачается около 20 Мб, займёт пару минут.

Если скачивание оборвалось на середине, удалите папку `quickdraw` и запустите ячейку ещё раз.

In [ ]:
CLASSES = ["apple", "banana", "cat", "fish", "guitar",
           "house", "moon", "airplane", "bicycle", "umbrella"]
N_TRAIN, N_TEST = 2000, 500
URL = "https://storage.googleapis.com/quickdraw_dataset/full/numpy_bitmap/{}.npy"


def download_drawings(name, n):
    # целиком файл весит около 100 Мб, поэтому читаем только первые n рисунков
    with urllib.request.urlopen(URL.format(name)) as r:
        version = np.lib.format.read_magic(r)
        if version == (1, 0):
            np.lib.format.read_array_header_1_0(r)
        else:
            np.lib.format.read_array_header_2_0(r)
        raw = r.read(n * 28 * 28)
    return np.frombuffer(raw, dtype=np.uint8).reshape(n, 28, 28)


def make_dataset(path="quickdraw"):
    if os.path.exists(path):
        print("Датасет уже скачан")
        return

    for part in ("train", "test"):
        os.makedirs(os.path.join(path, part))
        with open(os.path.join(path, part, "format.json"), "w") as fp:
            json.dump({name: i for i, name in enumerate(CLASSES)}, fp)

    for name in tqdm(CLASSES):
        drawings = download_drawings(name, N_TRAIN + N_TEST)
        for part, chunk in (("train", drawings[:N_TRAIN]), ("test", drawings[N_TRAIN:])):
            folder = os.path.join(path, part, name)
            os.makedirs(folder)
            for i, img in enumerate(chunk):
                Image.fromarray(img).save(os.path.join(folder, f"{i}.png"))


make_dataset()

In [ ]:
fig, axes = plt.subplots(3, 10, figsize=(14, 4.5))
for j, name in enumerate(CLASSES):
    for i in range(3):
        img = Image.open(f"quickdraw/train/{name}/{i}.png")
        axes[i, j].imshow(img, cmap="gray_r")
        axes[i, j].axis("off")
    axes[0, j].set_title(name)
plt.show()

---
## Часть 1. Pipeline обучения
Здесь почти всё как на семинаре, только данные другие.

### 1.1 Dataset

Напишите класс `DrawDataset`. Он должен:
- в `__init__` прочитать `format.json` и собрать список всех файлов вместе с их классами;
- в `__getitem__` открыть картинку, превратить её в вектор длины 784 со значениями от 0 до 1 и вернуть вместе с one-hot меткой (как на семинаре, через `torch.eye(10)`);
- в `__len__` вернуть количество картинок.

In [ ]:
class DrawDataset(data.Dataset):
    def __init__(self, path, train=True, transform=None):
        pass  # ваш код

    def __getitem__(self, item):
        pass  # ваш код

    def __len__(self):
        pass  # ваш код

### 1.2 DataLoader

Создайте обучающий датасет, разбейте его на train и validation в пропорции 0.8 / 0.2 (через `data.random_split` с генератором `torch.Generator().manual_seed(42)`) и сделайте три загрузчика:
- `train_data` с перемешиванием, `batch_size=32`;
- `train_data_val` без перемешивания, `batch_size=32`;
- `test_data` для тестовой выборки, `batch_size=500`.

In [ ]:
to_tensor = tfs.ToImage()

# ваш код

In [ ]:
# быстрая проверка
x, y = next(iter(train_data))
print(x.shape, y.shape)  # должно быть torch.Size([32, 784]) torch.Size([32, 10])
print(x.min().item(), x.max().item())  # от 0 до 1

### 1.3 Модель

Сеть такая же, как `DigitNet` на семинаре: линейный слой, ReLU, второй линейный слой.

In [ ]:
class DrawNet(nn.Module):
    def __init__(self, input_dim, num_hidden, output_dim):
        super().__init__()
        # ваш код

    def forward(self, x):
        pass  # ваш код

### 1.4 Цикл обучения как функция

 Дальше нам придётся обучать много разных моделей, поэтому оформите его в функцию `train_model`. Внутри можно почти дословно взять код с семинара.

Функция должна:
- на каждой эпохе обучать модель на `train_data` и считать средний loss (как `loss_mean` на семинаре);
- после каждой эпохи считать на `val_data` средний loss (как `Q_val`) и долю верных ответов;
- вернуть три списка: `loss_lst`, `loss_lst_val` и `acc_lst_val`.

Accuracy на валидации нам понадобится в частях 3 и 4: дальше будет видно, что loss и accuracy иногда рассказывают разные истории.

И ещё одна функция, `accuracy`, должна возвращать долю верных ответов модели на загрузчике (как мы считали `Q` на тесте).

In [ ]:
def train_model(model, optimizer, train_data, val_data, epochs, verbose=True):
    loss_function = nn.CrossEntropyLoss()
    loss_lst = []
    loss_lst_val = []
    acc_lst_val = []

    # ваш код

    return loss_lst, loss_lst_val, acc_lst_val


def accuracy(model, loader):
    pass  # ваш код

Теперь обучите `DrawNet(28 * 28, 128, 10)` с Adam (`lr=0.001`) 10 эпох, нарисуйте график loss на train и validation и посчитайте accuracy на тесте. Ориентир по accuracy: около 0.8 и выше. Если получается сильно меньше, скорее всего где-то ошибка.

In [ ]:
torch.manual_seed(42)

# ваш код

**Вопросы**
1. В `forward` мы не применяем softmax, хотя на выходе хотим вероятности классов. Почему так можно и даже нужно?
2. Посмотрите на картинки в начале ноутбука. Какие классы сеть, по-вашему, будет путать чаще всего и почему?
3. Сколько обучаемых параметров у `DrawNet(784, 128, 10)`? Посчитайте руками и проверьте кодом.

In [ ]:
# проверка числа параметров

**Ответы:**

1.

2.

3.

---
## Часть 2. Градиент руками

`loss.backward()` считает градиенты по цепному правилу. Давайте сделаем то же самое сами для одного слоя.

Берём линейный слой, после него ReLU, и функцию потерь MSE:

$$z = xW^T + b, \qquad h = \mathrm{ReLU}(z), \qquad L = \frac{1}{N \cdot d_{out}} \sum_{i,j} (h_{ij} - t_{ij})^2$$

Размеры: $x$ имеет размер $(N, d_{in})$, $W$ размер $(d_{out}, d_{in})$ (именно так веса хранит `nn.Linear`), $b$ размер $(d_{out})$, а $z$, $h$ и $t$ размер $(N, d_{out})$.

Нужно найти $\frac{\partial L}{\partial W}$, $\frac{\partial L}{\partial b}$ и $\frac{\partial L}{\partial x}$. Проще всего идти с конца: сначала $\frac{\partial L}{\partial h}$, потом $\frac{\partial L}{\partial z}$, а через него уже всё остальное.

### 2.1 Считаем руками

Напишите функцию `manual_grads`, которая делает прямой проход (z, h, loss) и считает `dW`, `db`, `dx` по вашим формулам. Вызывать `.backward()` и пользоваться `torch.autograd` здесь нельзя, только операции с матрицами.

In [ ]:
def manual_grads(x, W, b, t):
    # ваш код
    return dW, db, dx


torch.manual_seed(0)
x = torch.randn(5, 4)
W = torch.randn(3, 4)
b = torch.randn(3)
t = torch.randn(5, 3)

dW, db, dx = manual_grads(x, W, b, t)
print(dW.shape, db.shape, dx.shape)  # должны совпадать с W.shape, b.shape, x.shape

### 2.2 Сверяем с autograd

Теперь посчитайте те же градиенты через PyTorch: сделаем копии `x`, `W`, `b` с `requires_grad=True`, прогоним тот же прямой проход, вызовем `backward()` и сравним `.grad` с вашими `dW`, `db`, `dx`.

Сравнивать удобно через `torch.allclose(a, b, atol=1e-6)`. Все три сравнения должны дать `True`.

In [ ]:
x_ = x.clone().requires_grad_()
W_ = W.clone().requires_grad_()
b_ = b.clone().requires_grad_()

loss = ((F.relu(x_ @ W_.T + b_) - t) ** 2).mean()
loss.backward()

print("dW:", torch.allclose(dW, W_.grad, atol=1e-6))
print("db:", torch.allclose(db, b_.grad, atol=1e-6))
print("dx:", torch.allclose(dx, x_.grad, atol=1e-6))

### 2.3 Проверка на настоящей сети

Возьмите первый слой модели, которую обучили в части 1 (`model.layer1`), и батч картинок из `train_data_val`. Целевые значения `t` возьмите случайные, размером (32, 128).

Посчитайте loss как `F.mse_loss(F.relu(model.layer1(x_batch)), t_batch)`, вызовите `backward()` и сравните `model.layer1.weight.grad` и `model.layer1.bias.grad` с тем, что выдаёт ваша `manual_grads`. Перед этим не забудьте `model.zero_grad()`, чтобы там не остались градиенты от обучения.

In [ ]:
torch.manual_seed(1)

x_batch, _ = next(iter(train_data_val))
t_batch = torch.randn(32, 128)

# ваш код

**Вопросы**
1. ReLU не дифференцируема в нуле. Какую производную в этой точке берёт PyTorch? Проверьте маленьким экспериментом в ячейке ниже.
2. Зачем вообще считать $\frac{\partial L}{\partial x}$, если $x$ это данные и мы их не обучаем? Подумайте про сеть из нескольких слоёв.

In [ ]:
# эксперимент с производной ReLU в нуле

**Ответы:**

1.

2.

---
## Часть 3. Сравниваем оптимизаторы

Обучите одну и ту же `DrawNet(28 * 28, 128, 10)` тремя оптимизаторами по 15 эпох:

| Оптимизатор | Параметры |
|---|---|
| SGD | `lr=0.05` |
| SGD + momentum | `lr=0.05, momentum=0.9` |
| Adam | `lr=0.001` |

Чтобы сравнение было честным, перед созданием каждой модели вызывайте `torch.manual_seed(42)`. Тогда у всех трёх моделей будут одинаковые начальные веса и одинаковый порядок батчей.

Для каждой модели сохраните всё, что вернула `train_model`, и посчитайте accuracy на тесте. Функция `train_model` из части 1 уже умеет работать с любым оптимизатором, так что новый цикл писать не надо. Чтобы не засорять вывод, можно передать `verbose=False`.

In [ ]:
epochs = 15

configs = {
    "SGD": lambda params: # опитмизтор,
    "SGD + momentum": lambda params: # опитмизтор,
    "Adam": lambda params: # опитмизтор,
}

results = {}
for name, make_optimizer in configs.items():
  # ваш код

Посмотрим три графика, на каждом все три оптимизатора:

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
x_epochs = range(1, epochs + 1)

for name, (loss_lst, loss_lst_val, acc_lst_val) in results.items():
    axes[0].plot(x_epochs, loss_lst, label=name)
    axes[1].plot(x_epochs, loss_lst_val, label=name)
    axes[2].plot(x_epochs, acc_lst_val, label=name)
    print(f"{name}: минимальный val loss {min(loss_lst_val):.3f} на эпохе {np.argmin(loss_lst_val) + 1}")

axes[0].set_yscale("log")
for ax, title in zip(axes, ["loss на train", "loss на validation", "accuracy на validation"]):
    ax.set_title(title)
    ax.set_xlabel("эпоха")
    ax.grid(alpha=0.3)
    ax.legend()
plt.show()

<details>
<summary><b>Пояснение: почему у обычного SGD может оказаться самый низкий loss на валидации</b> (откройте после того, как посмотрите на графики)</summary>

<br>

Если у вас так получилось, это нормально, ошибки в коде нет.

**Adam и momentum не защищают от переобучения.** Это не регуляризация, а способ быстрее уменьшать loss на train. Чем быстрее оптимизатор учит сеть, тем раньше она начинает подстраиваться под конкретные картинки из обучающей выборки.

**Что происходит с Adam.** За несколько эпох он находит хорошее решение, а дальше сеть начинает переобучаться. Самое заметное последствие: она становится слишком уверенной в своих ответах. Раньше про картинку кота она говорила «кот с вероятностью 0.7», а теперь «кот с вероятностью 0.999».

Посмотрим, как на это реагирует кросс-энтропия. Loss на одном объекте равен $-\log p$, где $p$ это вероятность, которую сеть дала правильному классу:
- сеть правильно ответила с вероятностью 0.7: loss примерно 0.36;
- правильно с вероятностью 0.999: loss примерно 0.001. Выигрыш маленький;
- ошиблась, но неуверенно, правильному классу дала 0.3: loss примерно 1.2;
- ошиблась уверенно, правильному классу дала 0.001: loss примерно 6.9. Штраф огромный.

Так что пара уверенных ошибок поднимает средний loss на валидации, хотя доля правильных ответов почти не меняется. Поэтому у Adam val loss растёт, а val accuracy остаётся высокой.

**Что происходит с обычным SGD.** Он учит медленно. К 15-й эпохе он ещё не доучился и до переобучения не дошёл. Уверенных ошибок у него мало, поэтому val loss низкий. Но и правильных ответов меньше, отсюда худшая accuracy на тесте.

**Вывод.** Сравнивать модели по val loss в последней эпохе опасно, картина может получиться обманчивой. Смотрите на кривые целиком и на accuracy: на графике val accuracy видно, что SGD отстаёт. А эпоха, на которой val loss у Adam был минимальным, показывает, где обучение стоило остановить. Этот приём называется ранней остановкой (early stopping).

</details>

**Вопросы**
1. Какой оптимизатор быстрее всех уменьшает loss на train? А какой даёт лучший результат на validation в конце?
2. Объясните своими словами, что делает momentum и почему он помогает обычному SGD. Видно ли это на ваших графиках?
3. Если бы вам дали новую задачу и одну попытку обучения, какой оптимизатор вы бы взяли первым и почему?

**Ответы:**

1.

2.

3.

---
## Часть 4. Dropout и BatchNorm

Чтобы было что лечить, сначала устроим переобучение. Оставим для обучения только 500 картинок и возьмём сеть побольше: два скрытых слоя по 512 нейронов. Такая сеть быстро выучит эти 500 картинок наизусть.

Потом добавим Dropout и BatchNorm и посмотрим, что из них на самом деле помогает от переобучения. Заранее не ждите, что помогут оба одинаково: про эти два слоя часто думают, что они делают одно и то же, но это не так.

In [ ]:
d_small = data.Subset(d_train, range(500))
train_small = data.DataLoader(d_small, shuffle=True, batch_size=32)
print(len(d_small))

### 4.1 Модель

Напишите `DrawNetReg` с тремя линейными слоями: `input_dim > num_hidden > num_hidden > output_dim`. После каждого из двух скрытых слоёв идёт:
- BatchNorm (`nn.BatchNorm1d`), если `use_bn=True`;
- ReLU;
- Dropout с вероятностью `p_drop` (при `p_drop=0` он просто ничего не делает).

После последнего слоя ничего не добавляем.

In [ ]:
class DrawNetReg(nn.Module):
    def __init__(self, input_dim, num_hidden, output_dim, p_drop=0.0, use_bn=False):
        super().__init__()
        # ваш код

    def forward(self, x):
        pass  # ваш код

### 4.2 Четыре эксперимента

Обучите четыре варианта `DrawNetReg(28 * 28, 512, 10, ...)` на `train_small` по 60 эпох, валидация по-прежнему на `train_data_val`. Оптимизатор Adam, `lr=0.001`, перед каждой моделью `torch.manual_seed(42)`.

| Вариант | p_drop | use_bn |
|---|---|---|
| baseline | 0 | False |
| dropout | 0.5 | False |
| batchnorm | 0 | True |
| dropout + bn | 0.5 | True |

Для каждого варианта нарисуйте график loss на train и validation и отдельно accuracy на validation (удобно сделать сетку 2 на 4: сверху loss, снизу accuracy) и посчитайте accuracy на тесте. Обученные модели сохраните, одна понадобится в 4.3.

In [ ]:
epochs = 60

variants = {
    "baseline": dict(#параметры модели),
    "dropout": dict(#параметры модели),
    "batchnorm": dict(#параметры модели),
    "dropout + bn": dict(#параметры модели),
}

reg_results = {}
reg_models = {}

# Ваш код

fig, axes = plt.subplots(2, 4, figsize=(18, 7), sharey="row")
for j, (name, (loss_lst, loss_lst_val, acc_lst_val)) in enumerate(reg_results.items()):
    axes[0, j].plot(loss_lst, label="train")
    axes[0, j].plot(loss_lst_val, label="val")
    axes[0, j].set_title(name)
    axes[0, j].legend()
    axes[1, j].plot(acc_lst_val, color="tab:green")
    axes[1, j].set_xlabel("эпоха")
    for ax in axes[:, j]:
        ax.grid(alpha=0.3)
axes[0, 0].set_ylabel("loss")
axes[1, 0].set_ylabel("accuracy на validation")
plt.show()

<details>
<summary><b>Пояснение: почему BatchNorm может оказаться хуже baseline и как Dropout уживается с BatchNorm</b> (откройте после того, как посмотрите на графики)</summary>

<br>

**BatchNorm хуже baseline.** BatchNorm придумывали не для борьбы с переобучением. Его главная задача в том, чтобы ускорить и стабилизировать обучение. Он нормирует выходы слоя, и на вход следующему слою всегда приходят значения примерно одного масштаба. Из-за этого сеть учится быстрее и меньше зависит от неудачной инициализации или learning rate.

Небольшой регуляризующий эффект у BatchNorm тоже есть. Среднее и дисперсия считаются по случайному батчу, и это добавляет в обучение немного шума. Но эффект слабый и побочный.

У нас всего 500 картинок. Сеть с BatchNorm запоминает их ещё быстрее, чем baseline: train loss почти сразу уходит к нулю, а val loss начинает расти уже через несколько эпох. Сравните, как быстро падает train loss у baseline и у batchnorm. Это и есть ускорение, которое даёт BatchNorm. А по-настоящему от переобучения в этом эксперименте помогает Dropout.

**Dropout вместе с BatchNorm.** Использовать их вместе не запрещено, но есть известный эффект. Его подробно разобрали в статье [Li et al., 2019, «Understanding the Disharmony between Dropout and Batch Normalization by Variance Shift»](https://arxiv.org/abs/1801.05134).

Чтобы понять суть, вспомним, как работают оба слоя:
- BatchNorm во время обучения копит в `running_mean` и `running_var` среднее и дисперсию своих входов. В режиме `eval()` он нормирует данные именно по этим накопленным значениям;
- Dropout во время обучения обнуляет часть нейронов, а оставшиеся умножает на $\frac{1}{1-p}$, чтобы среднее не изменилось. Среднее действительно сохраняется, а вот разброс значений становится больше. В режиме `eval()` Dropout выключен, и разброс снова обычный.

Теперь представим, что Dropout стоит перед BatchNorm. Во время обучения BatchNorm видит данные с увеличенным разбросом и запоминает именно такую дисперсию. При предсказании Dropout выключается, разброс входов становится меньше, а BatchNorm продолжает нормировать по старой, завышенной дисперсии. Получается, что он подстроен под одни данные, а работает с другими. В нашей сети так устроен второй блок: Dropout после первого слоя стоит прямо перед BatchNorm второго слоя.

Сильнее всего это мешает глубоким свёрточным сетям, где таких мест много и ошибка накапливается от слоя к слою. В небольшой сети вроде нашей связка обычно работает нормально, что и видно в эксперименте. Если хочется подстраховаться, Dropout ставят только после последнего слоя с BatchNorm, ближе к выходу сети.

</details>

### 4.3 Что будет, если забыть `model.eval()`

Возьмите модель `dropout` из прошлого пункта, переведите её в режим `model.train()` и посчитайте accuracy на тесте три раза подряд. Функция `accuracy` здесь не подойдёт, потому что она сама включает `eval()`, так что посчитайте вручную (не забудьте `torch.no_grad()`). Потом сравните с accuracy в режиме `eval()`.

In [ ]:
# ваш код

**Вопросы**
1. Примерно с какой эпохи у baseline начинается переобучение? По каким признакам на графике вы это поняли?
2. Какой вариант сильнее всего уменьшил разрыв между train и validation? Какой дал лучшую accuracy на тесте? Это один и тот же вариант?
3. У модели с dropout loss на train может оказаться выше, чем на validation. Как такое может быть?

**Ответы:**

1.

2.

3.